# Datacube Loading and Moment Maps

This notebook demonstrates the first runnable workflow in the project: generating a small synthetic molecular-line FITS datacube, loading it with `spectral-cube`, extracting a spectrum, and making moment maps.

## What is a spectral-line datacube?

A molecular spectral-line datacube stores intensity as a function of two sky coordinates and one spectral coordinate. Each spatial pixel contains a spectrum. For radio and millimeter astronomy, the spectral axis is often represented as frequency or Doppler velocity relative to a molecular rest frequency.

## Why synthetic data here?

The repository should run before any public telescope data are downloaded. A synthetic cube lets us test the workflow, WCS metadata, extraction functions, and moment maps in a reproducible way. Later milestones can swap this cube for real observations.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent.resolve()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import matplotlib.pyplot as plt
from astropy import units as u

from molecular_conditions.datacube import load_fits_cube, summarize_cube_metadata
from molecular_conditions.demo_data import create_synthetic_line_cube
from molecular_conditions.moments import make_moment0, make_moment1, make_moment2, moment0_to_k_kms
from molecular_conditions.spectra import extract_pixel_spectrum, extract_region_spectrum, estimate_rms_noise

DATA_DIR = PROJECT_ROOT / "data" / "demo"
FIGURE_DIR = PROJECT_ROOT / "figures"
DATA_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

## Generate and load the synthetic cube

The cube contains one Gaussian line with a spatially varying amplitude, a simple velocity gradient, and Gaussian noise. Its spectral axis is radio velocity in m/s.

In [ ]:
cube_path = create_synthetic_line_cube(DATA_DIR / "synthetic_line_cube.fits", overwrite=True)
cube = load_fits_cube(cube_path)
summary = summarize_cube_metadata(cube)

for key, value in summary.items():
    print(f"{key}: {value}")

## Extract a spectrum

The central pixels have the strongest synthetic emission. The line-free window on the blue side of the cube is used to estimate the RMS noise.

In [ ]:
spectral_axis, intensity = extract_pixel_spectrum(cube, x=16, y=15)
rms = estimate_rms_noise((spectral_axis, intensity), line_free_window=(-6300.0, -4300.0))

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(spectral_axis.to_value(u.km / u.s), intensity.value, color="black", lw=1.5)
ax.axhline(rms, color="tab:red", ls="--", lw=1, label=f"RMS = {rms:.3f} K")
ax.axhline(-rms, color="tab:red", ls="--", lw=1)
ax.set_xlabel("Radio velocity (km/s)")
ax.set_ylabel(f"Intensity ({intensity.unit})")
ax.set_title("Synthetic molecular-line spectrum")
ax.legend()
fig.tight_layout()
fig.savefig(FIGURE_DIR / "synthetic_pixel_spectrum.png", dpi=150)
plt.show()

## Region-averaged spectra

Real molecular cloud observations often have low signal-to-noise in individual pixels. Averaging spectra over a physically motivated region, such as a dense core, outflow lobe, or integrated-intensity contour, can improve line detection and provide a more representative spectrum for line fitting.

In [ ]:
region = {"x_min": 12, "x_max": 20, "y_min": 11, "y_max": 19}
region_axis, region_intensity = extract_region_spectrum(cube, region, statistic="mean")

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(region_axis.to_value(u.km / u.s), region_intensity.value, color="tab:blue", lw=1.8)
ax.set_xlabel("Radio velocity (km/s)")
ax.set_ylabel(f"Mean intensity ({region_intensity.unit})")
ax.set_title("Region-averaged synthetic spectrum")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "synthetic_region_average_spectrum.png", dpi=150)
plt.show()

## Moment maps

- **Moment 0** is integrated intensity. It shows where the molecular emission is bright.
- **Moment 1** is the intensity-weighted velocity centroid. It traces the bulk velocity field.
- **Moment 2** is the velocity dispersion, or line-width sigma. It traces broadening from thermal, turbulent, and unresolved kinematic structure.

The cube's native velocity axis is in m/s, so the internal moment 0 calculation naturally gives K m/s. For display, we convert this to the more common radio-astronomy unit K km/s.

In [ ]:
window = (-3500.0, 3500.0)
moment0 = moment0_to_k_kms(make_moment0(cube, window))
moment1 = make_moment1(cube, window).to(u.km / u.s)
moment2 = make_moment2(cube, window).to(u.km / u.s)

maps = [
    (moment0, "Moment 0", str(moment0.unit)),
    (moment1, "Moment 1", "km/s"),
    (moment2, "Moment 2", "km/s"),
]

fig, axes = plt.subplots(1, 3, figsize=(12, 3.8), constrained_layout=True)
for ax, (image, title, label) in zip(axes, maps):
    im = ax.imshow(image.value, origin="lower", cmap="viridis")
    ax.set_title(title)
    ax.set_xlabel("x pixel")
    ax.set_ylabel("y pixel")
    fig.colorbar(im, ax=ax, shrink=0.82, label=label)

fig.savefig(FIGURE_DIR / "synthetic_moment_maps.png", dpi=150)
plt.show()